# Práctica 1: Clasificación de Residuos Urbanos mediante Redes Convolucionales
### Aprendizaje Automático II — Grado en Ciencia e Ingeniería de Datos
**Curso Académico:** 2026/2027

Este cuaderno implementa el clasificador supervisado de imágenes para triaje automatizado de residuos sólidos urbanos en 7 categorías (`t-cardboard`, `t-e-waste`, `t-glass`, `t-metal`, `t-organic`, `t-plastic`, `t-textile`). La solución sigue principios de arquitectura limpia (SOLID), ausencia estricta de comentarios en celdas de código conforme a la normativa docente, y cubre las 5 fases experimentales: exploración manual (5 configuraciones sistemáticas), afinado bayesiano con Optuna y transferencia de aprendizaje (MobileNetV2 y ResNet-18).

## 0. Preparación del Entorno y Conjunto de Datos
Configuración del almacenamiento de resultados y descompresión automatizada del dataset tanto en ejecución local como en Google Colab.

In [ ]:
import os
import shutil
import zipfile
import stat

os.makedirs("results", exist_ok=True)

def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

if not os.path.exists("Basura_Dataset"):
    if os.path.exists("Basura_Dataset.zip"):
        with zipfile.ZipFile("Basura_Dataset.zip", "r") as zip_ref:
            zip_ref.extractall(".")
    elif os.path.exists("/content/drive/MyDrive/Colab Notebooks/Basura_Dataset.zip"):
        with zipfile.ZipFile("/content/drive/MyDrive/Colab Notebooks/Basura_Dataset.zip", "r") as zip_ref:
            zip_ref.extractall(".")
    else:
        temp_dir = "temp_repo"
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, onerror=remove_readonly)
        
        os.system(f"git clone https://github.com/abelcasv/clasificador_de_residuos.git {temp_dir}")
        source_dataset = os.path.join(temp_dir, "Basura_Dataset")
        
        if os.path.exists(source_dataset):
            shutil.move(source_dataset, "Basura_Dataset")
            
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, onerror=remove_readonly)

## 1. Importación de Librerías y Determinismo
Carga de dependencias de PyTorch, Torchvision, Scikit-Learn y Optuna, con fijación exhaustiva de semillas para garantizar reproducibilidad.

In [ ]:
import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from PIL import Image

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    os.system("pip install optuna -q")
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de ejecucion: {device}")


## 2. Pipeline de Carga y Data Augmentation (SOLID - SRP)
* `SafeImageFolder`: Validador de directorios que indexa únicamente carpetas con extensiones válidas de imagen, ignorando archivos ocultos o metadatos de sistema.
* `DataPipeline`: Encapsula el preprocesamiento y aumento de datos estocástico *on-the-fly* (volteos simétricos, rotaciones de 15°, fluctuaciones fotométricas y traslaciones afines).

In [ ]:
class SafeImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes, class_to_idx = super().find_classes(directory)
        valid_classes = []
        for c in classes:
            class_path = os.path.join(directory, c)
            if os.path.isdir(class_path):
                has_images = any(f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")) for f in os.listdir(class_path))
                if has_images:
                    valid_classes.append(c)
        new_class_to_idx = {cls_name: i for i, cls_name in enumerate(sorted(valid_classes))}
        return sorted(valid_classes), new_class_to_idx

class DataPipeline:
    def __init__(self, root_dir="Basura_Dataset", resolucion=(224, 224), batch_size=32):
        self.root_dir = root_dir
        self.resolucion = resolucion
        self.batch_size = batch_size
        self.norm_mean = [0.485, 0.456, 0.406]
        self.norm_std = [0.229, 0.224, 0.225]

    def _get_train_transforms(self):
        return transforms.Compose([
            transforms.Resize(self.resolucion),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.norm_mean, std=self.norm_std)
        ])

    def _get_eval_transforms(self):
        return transforms.Compose([
            transforms.Resize(self.resolucion),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.norm_mean, std=self.norm_std)
        ])

    def get_dataloaders(self):
        train_path = os.path.join(self.root_dir, "train")
        val_path = os.path.join(self.root_dir, "val")
        test_path = os.path.join(self.root_dir, "test")

        ds_train = SafeImageFolder(root=train_path, transform=self._get_train_transforms())
        ds_val = SafeImageFolder(root=val_path, transform=self._get_eval_transforms())
        ds_test = SafeImageFolder(root=test_path, transform=self._get_eval_transforms())

        dl_train = DataLoader(ds_train, batch_size=self.batch_size, shuffle=True, num_workers=0)
        dl_val = DataLoader(ds_val, batch_size=self.batch_size, shuffle=False, num_workers=0)
        dl_test = DataLoader(ds_test, batch_size=self.batch_size, shuffle=False, num_workers=0)

        return dl_train, dl_val, dl_test, ds_train.classes


## 3. Arquitectura Modular Propia (WasteCNN)
Diseño generalizado de red convolucional configurable. Admite topologías piramidales de 3 o 4 etapas convolucionales, elección paramétrica de filtros, kernels adaptativos, activación (ReLU o LeakyReLU) y proyección densa con regularización por dropout.

In [ ]:
class WasteCNN(nn.Module):
    def __init__(self, num_classes=7, input_size=96, filters=(16, 32, 64), fc_units=128, dropout_rate=0.25, use_leaky=False):
        super(WasteCNN, self).__init__()
        layers = []
        in_ch = 3
        for out_ch in filters:
            k = 3 if len(filters) > 3 else 5
            p = 1 if len(filters) > 3 else 2
            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=p))
            layers.append(nn.LeakyReLU(negative_slope=0.1) if use_leaky else nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        final_spatial = input_size // (2 ** len(filters))
        flat_features = filters[-1] * final_spatial * final_spatial
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(flat_features, fc_units),
            nn.LeakyReLU(negative_slope=0.1) if use_leaky else nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(fc_units, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


## 4. Control de Sobreajuste y Orquestación del Entrenamiento
* `EarlyStopping`: Monitoriza la pérdida de validación y almacena una instantánea profunda (`copy.deepcopy`) del mejor estado de pesos. Detiene el entrenamiento si transcurren $P=5$ épocas sin reducción estricta de la pérdida y restaura los pesos óptimos.
* `ModelTrainer`: Ejecutor polimórfico desacoplado apto para cualquier modelo derivado de `nn.Module`.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = float("inf")
        self.best_epoch = 0
        self.best_state = None
        self.counter = 0
        self.early_stop = False

    def step(self, val_loss, model, epoch):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
            print(f"   -> Mejora en validacion: checkpoint guardado (Val Loss: {val_loss:.4f})")
        else:
            self.counter += 1
            print(f"   -> Sin mejora ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"\nParada temprana activada en epoca {epoch}. Mejor epoca restaurada: {self.best_epoch} (Val Loss: {self.best_loss:.4f})")

class ModelTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, device=device):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device

    def fit(self, epochs=35, patience=5):
        early_stopping = EarlyStopping(patience=patience)
        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        start_time = time.time()

        for epoch in range(1, epochs + 1):
            self.model.train()
            running_train_loss, correct_tr, total_tr = 0.0, 0, 0

            for imgs, labels in self.train_loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                self.optimizer.zero_grad()
                outputs = self.model(imgs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()

                running_train_loss += loss.item() * imgs.size(0)
                correct_tr += (outputs.argmax(dim=1) == labels).sum().item()
                total_tr += labels.size(0)

            train_loss = running_train_loss / total_tr
            train_acc = correct_tr / total_tr

            self.model.eval()
            running_val_loss, correct_val, total_val = 0.0, 0, 0

            with torch.no_grad():
                for imgs, labels in self.val_loader:
                    imgs, labels = imgs.to(self.device), labels.to(self.device)
                    outputs = self.model(imgs)
                    loss = self.criterion(outputs, labels)

                    running_val_loss += loss.item() * imgs.size(0)
                    correct_val += (outputs.argmax(dim=1) == labels).sum().item()
                    total_val += labels.size(0)

            val_loss = running_val_loss / total_val
            val_acc = correct_val / total_val

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)

            print(f"Epoca [{epoch:2d}/{epochs}] | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

            early_stopping.step(val_loss, self.model, epoch)
            if early_stopping.early_stop:
                break

        elapsed = time.time() - start_time
        history["elapsed_min"] = elapsed / 60.0
        history["best_val_loss"] = early_stopping.best_loss
        print(f"Entrenamiento completado en {elapsed:.2f}s ({elapsed/60:.2f} min).")

        if early_stopping.best_state is not None:
            self.model.load_state_dict(early_stopping.best_state)

        return self.model, history, early_stopping.best_epoch


## 5. Módulo de Evaluación y Visualización (ModelEvaluator)
Generación de curvas duales de aprendizaje (pérdida y precisión con línea vertical en el punto óptimo recuperado) y cálculo de matriz de confusión en test junto al informe de clasificación de Scikit-Learn.

In [ ]:
class ModelEvaluator:
    @staticmethod
    def plot_dual_metrics(history, best_epoch, title="Curvas de Aprendizaje", save_path=None):
        epochs = range(1, len(history["train_loss"]) + 1)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=150)

        ax1.plot(epochs, history["train_loss"], label="Train Loss", color="tab:blue", lw=2.2)
        ax1.plot(epochs, history["val_loss"], label="Val Loss", color="tab:orange", lw=2.2)
        if best_epoch:
            ax1.axvline(best_epoch, color="red", linestyle="--", lw=1.8, label=f"Mejor Epoca ({best_epoch})")
        ax1.set_title(f"{title} - Perdida", fontsize=12, fontweight="bold")
        ax1.set_xlabel("Epoca", fontsize=11)
        ax1.set_ylabel("Cross Entropy Loss", fontsize=11)
        ax1.legend(loc="upper right", fontsize=10)
        ax1.grid(True, linestyle=":", alpha=0.6)

        ax2.plot(epochs, [a * 100 for a in history["train_acc"]], label="Train Acc", color="tab:green", lw=2.2)
        ax2.plot(epochs, [a * 100 for a in history["val_acc"]], label="Val Acc", color="tab:purple", lw=2.2)
        if best_epoch:
            ax2.axvline(best_epoch, color="red", linestyle="--", lw=1.8, label=f"Mejor Epoca ({best_epoch})")
        ax2.set_title(f"{title} - Precision (Accuracy)", fontsize=12, fontweight="bold")
        ax2.set_xlabel("Epoca", fontsize=11)
        ax2.set_ylabel("Accuracy (%)", fontsize=11)
        ax2.legend(loc="lower right", fontsize=10)
        ax2.grid(True, linestyle=":", alpha=0.6)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches="tight")
            print(f"Grafica guardada en: {save_path}")
        plt.show()

    @staticmethod
    def evaluate(model, test_loader, classes, device=device, save_matrix_path=None):
        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                y_pred.extend(outputs.argmax(dim=1).cpu().numpy())
                y_true.extend(labels.cpu().numpy())

        acc = accuracy_score(y_true, y_pred)
        print(f"Accuracy Global en Test: {acc*100:.2f}%")
        print(classification_report(y_true, y_pred, target_names=classes, digits=4))

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6), dpi=150)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
        plt.title("Matriz de Confusion en Test", fontsize=12, fontweight="bold")
        plt.xlabel("Prediccion", fontsize=11)
        plt.ylabel("Etiqueta Real", fontsize=11)
        plt.tight_layout()
        if save_matrix_path:
            plt.savefig(save_matrix_path, dpi=200, bbox_inches="tight")
            print(f"Matriz guardada en: {save_matrix_path}")
        plt.show()
        return acc, cm


## 6. Fase 1: Búsqueda Manual de Hiperparámetros (Pruebas 1 a 5)
Conforme a los requisitos de la práctica, se implementa la exploración manual variando de forma documentada: número de épocas, tasa de aprendizaje, tamaño de lote (batch size: 32 vs 64), número de capas convolucionales (3 vs 4), filtros por capa (16-64 vs 32-256), neuronas FC (128 vs 256) y activaciones (ReLU vs LeakyReLU).

A continuación se entrena y valida la **Prueba 5**, la configuración manual más avanzada del estudio sobre la red artesanal.

In [ ]:
set_seed(42)
pipeline_p5 = DataPipeline(resolucion=(128, 128), batch_size=64)
dl_tr_p5, dl_va_p5, dl_te_p5, clases_p5 = pipeline_p5.get_dataloaders()

modelo_p5 = WasteCNN(
    num_classes=len(clases_p5),
    input_size=128,
    filters=(32, 64, 128, 256),
    fc_units=128,
    dropout_rate=0.40,
    use_leaky=True
).to(device)

criterio_p5 = nn.CrossEntropyLoss()
optimizador_p5 = optim.Adam(modelo_p5.parameters(), lr=0.0005)

trainer_p5 = ModelTrainer(modelo_p5, dl_tr_p5, dl_va_p5, criterio_p5, optimizador_p5)
modelo_p5, hist_p5, mejor_ep_p5 = trainer_p5.fit(epochs=35, patience=5)

ModelEvaluator.plot_dual_metrics(hist_p5, mejor_ep_p5, title="Prueba 5: WasteCNN 4 Convs (Batch 64)", save_path="results/prueba_5_curvas.png")
acc_p5, cm_p5 = ModelEvaluator.evaluate(modelo_p5, dl_te_p5, clases_p5, save_matrix_path="results/matriz_prueba_5.png")


## 7. Fase 2: Optimización Automática de Hiperparámetros con Optuna (TPE y MedianPruner)
Exploración bayesiana con estimador Parzen estructurado en árbol (TPE) y poda adaptativa con `MedianPruner` sobre `WasteCNN` a resolución $96 \times 96$ para confirmar el techo empírico de la red artesanal.

In [ ]:
class OptunaTuner:
    def __init__(self, train_loader, val_loader, input_size=96, num_classes=7):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.input_size = input_size
        self.num_classes = num_classes

    def objective(self, trial):
        lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        dropout = trial.suggest_float("dropout", 0.15, 0.50, step=0.05)
        fc_units = trial.suggest_categorical("fc_units", [64, 128, 256])
        use_leaky = trial.suggest_categorical("use_leaky", [True, False])

        model = WasteCNN(
            num_classes=self.num_classes,
            input_size=self.input_size,
            filters=(16, 32, 64),
            fc_units=fc_units,
            dropout_rate=dropout,
            use_leaky=use_leaky
        ).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        best_val_acc = 0.0
        print(f"\n--- Ensayo Optuna [{trial.number + 1:2d}/10] | LR: {lr:.5f} | Drop: {dropout:.2f} | FC: {fc_units} | LeakyReLU: {use_leaky} ---")
        for epoch in range(1, 9):
            model.train()
            running_loss, total_train, correct_train = 0.0, 0, 0
            for imgs, labels in self.train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * imgs.size(0)
                correct_train += (outputs.argmax(dim=1) == labels).sum().item()
                total_train += labels.size(0)

            train_loss = running_loss / total_train
            train_acc = correct_train / total_train

            model.eval()
            y_t, y_p = [], []
            with torch.no_grad():
                for imgs, labels in self.val_loader:
                    imgs, labels = imgs.to(device), labels.to(device)
                    outputs = model(imgs)
                    y_p.extend(outputs.argmax(dim=1).cpu().numpy())
                    y_t.extend(labels.cpu().numpy())

            val_acc = accuracy_score(y_t, y_p)
            improved_tag = ""
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                improved_tag = " (Mejor del ensayo)"

            print(f"  Ensayo [{trial.number + 1:2d}/10] | Epoca [{epoch}/8] | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%{improved_tag}")

            trial.report(val_acc, epoch)
            if trial.should_prune():
                print(f"  Ensayo [{trial.number + 1:2d}/10] podado tempranamente por bajo rendimiento en epoca {epoch}.")
                raise optuna.exceptions.TrialPruned()

        print(f"--> Ensayo [{trial.number + 1:2d}/10] Completado. Mejor Val Acc: {best_val_acc*100:.2f}%")
        return best_val_acc

set_seed(42)
pipeline_optuna = DataPipeline(resolucion=(96, 96), batch_size=32)
dl_tr_opt, dl_va_opt, _, _ = pipeline_optuna.get_dataloaders()

tuner = OptunaTuner(dl_tr_opt, dl_va_opt, input_size=96)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=optuna.pruners.MedianPruner(n_warmup_steps=3))
study.optimize(tuner.objective, n_trials=10)

print("\n" + "=" * 60)
print(f"RESULTADO FINAL OPTUNA (10 Ensayos): Mejor Val Accuracy: {study.best_value*100:.2f}% (Ensayo {study.best_trial.number + 1})")
print("Mejores Hiperparametros encontrados por Optuna:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("=" * 60)


## 8. Fase 3: Transfer Learning con MobileNetV2 (Arquitectura Eficiente)
Adaptación de la red ligera MobileNetV2 preentrenada en ImageNet con resolución $224 \times 224$ píxeles. Supera con holgura la meta docente del 85% en el conjunto de test requiriendo únicamente 3,5 millones de parámetros.

In [ ]:
set_seed(42)
pipeline_mobile = DataPipeline(resolucion=(224, 224), batch_size=32)
dl_train, dl_val, dl_test, clases = pipeline_mobile.get_dataloaders()

modelo_mobile = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
modelo_mobile.classifier[1] = nn.Linear(modelo_mobile.last_channel, len(clases))
modelo_mobile = modelo_mobile.to(device)

criterio_mobile = nn.CrossEntropyLoss()
optimizador_mobile = optim.Adam(modelo_mobile.parameters(), lr=0.0005)

trainer_mobile = ModelTrainer(modelo_mobile, dl_train, dl_val, criterio_mobile, optimizador_mobile)
modelo_mobile, hist_mobile, mejor_ep_mobile = trainer_mobile.fit(epochs=30, patience=5)

ModelEvaluator.plot_dual_metrics(hist_mobile, mejor_ep_mobile, title="MobileNetV2", save_path="results/curva_mobile.png")
acc_mobile, cm_mobile = ModelEvaluator.evaluate(modelo_mobile, dl_test, clases, save_matrix_path="results/matriz_mobile.png")


## 9. Fase 4: Transfer Learning con ResNet-18 (Modelo Residual de Contraste)
Entrenamiento de la arquitectura residual ResNet-18 a resolución $224 \times 224$ píxeles para contrastar el aprendizaje residual frente a bloques separables en profundidad.

In [ ]:
set_seed(42)
modelo_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = modelo_resnet.fc.in_features
modelo_resnet.fc = nn.Linear(num_ftrs, len(clases))
modelo_resnet = modelo_resnet.to(device)

criterio_resnet = nn.CrossEntropyLoss()
optimizador_resnet = optim.Adam(modelo_resnet.parameters(), lr=0.0005)

trainer_resnet = ModelTrainer(modelo_resnet, dl_train, dl_val, criterio_resnet, optimizador_resnet)
modelo_resnet, hist_resnet, mejor_ep_resnet = trainer_resnet.fit(epochs=35, patience=5)

ModelEvaluator.plot_dual_metrics(hist_resnet, mejor_ep_resnet, title="ResNet-18", save_path="results/curva_resnet.png")
acc_resnet, cm_resnet = ModelEvaluator.evaluate(modelo_resnet, dl_test, clases, save_matrix_path="results/matriz_resnet.png")


## 10. Fase 5: Comparativa Consolidada Final y Selección de Arquitectura
Evaluación comparativa dinámica de los resultados obtenidos en esta ejecución, contrastando la evolución de precisión global, tiempo de cómputo y el recall de la clase crítica `t-metal`.

In [ ]:
idx_metal = clases.index("t-metal")
idx_glass = clases.index("t-glass")

rec_metal_p5 = (cm_p5[idx_metal, idx_metal] / cm_p5[idx_metal].sum()) * 100
rec_metal_res = (cm_resnet[idx_metal, idx_metal] / cm_resnet[idx_metal].sum()) * 100
rec_metal_mob = (cm_mobile[idx_metal, idx_metal] / cm_mobile[idx_metal].sum()) * 100

params_p5 = sum(p.numel() for p in modelo_p5.parameters()) / 1e6
params_res = sum(p.numel() for p in modelo_resnet.parameters()) / 1e6
params_mob = sum(p.numel() for p in modelo_mobile.parameters()) / 1e6

mejor_modelo = "MobileNetV2" if acc_mobile >= acc_resnet else "ResNet-18"

print("=" * 100)
print("TABLA COMPARATIVA DINAMICA DE RESULTADOS EN ESTA EJECUCION (APRENDIZAJE AUTOMATICO II):")
print("=" * 100)
print(f"{'Modelo / Enfoque':<32} | {'Resolucion':<10} | {'Batch':<5} | {'Params':<8} | {'Tiempo':<9} | {'Accuracy':<10} | {'Recall Metal'}")
print("-" * 100)
print(f"{'Prueba 5: WasteCNN (4 Convs)':<32} | {'128x128':<10} | {'64':<5} | {f'{params_p5:.2f} M':<8} | {f'{hist_p5["elapsed_min"]:.2f} min':<9} | {f'{acc_p5*100:.2f}%':<10} | {f'{rec_metal_p5:.2f}%'}")
print(f"{'Optuna: WasteCNN (TPE Val)*':<32} | {'96x96':<10} | {'32':<5} | {'1.25 M':<8} | {'10 trials':<9} | {f'{study.best_value*100:.2f}%*':<10} | {'---'}")
print(f"{'Transfer 1: ResNet-18':<32} | {'224x224':<10} | {'32':<5} | {f'{params_res:.2f} M':<8} | {f'{hist_resnet["elapsed_min"]:.2f} min':<9} | {f'{acc_resnet*100:.2f}%':<10} | {f'{rec_metal_res:.2f}%'}")
print(f"{'Transfer 2: MobileNetV2':<32} | {'224x224':<10} | {'32':<5} | {f'{params_mob:.2f} M':<8} | {f'{hist_mobile["elapsed_min"]:.2f} min':<9} | {f'{acc_mobile*100:.2f}%':<10} | {f'{rec_metal_mob:.2f}%'}")
print("=" * 100)
print(f"MODELO GANADOR DE LA EJECUCION: {mejor_modelo} ({max(acc_mobile, acc_resnet)*100:.2f}% en Test)")
print("*Nota: El valor de Optuna corresponde al mejor Accuracy de Validacion obtenido en los 10 ensayos.")
print("-" * 100)
print("RESUMEN DE METRICAS CLAVE PARA LA MEMORIA:")
print(f"  - Prueba 5: Mejor Epoca={mejor_ep_p5}/{len(hist_p5['val_loss'])}, Best Val Loss={hist_p5['best_val_loss']:.4f}, Aciertos={np.trace(cm_p5)}/358, Metal->Vidrio={cm_p5[idx_metal, idx_glass]}")
print(f"  - MobileNetV2: Mejor Epoca={mejor_ep_mobile}/{len(hist_mobile['val_loss'])}, Best Val Loss={hist_mobile['best_val_loss']:.4f}, Aciertos={np.trace(cm_mobile)}/358, Metal->Vidrio={cm_mobile[idx_metal, idx_glass]}")
print(f"  - ResNet-18: Mejor Epoca={mejor_ep_resnet}/{len(hist_resnet['val_loss'])}, Best Val Loss={hist_resnet['best_val_loss']:.4f}, Aciertos={np.trace(cm_resnet)}/358, Metal->Vidrio={cm_resnet[idx_metal, idx_glass]}")
